In [1]:
import nltk
import pandas as pd


In [2]:
dataset=pd.read_csv('/home/hammadali08/Personal/CSV file/IMDB Dataset.csv')
dataset

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [3]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()
corpus=[]
stop_words = set([w.lower() for w in stopwords.words('english')])

for i in range(0, len(dataset)):
    review = re.sub('[^a-zA-Z0-9]', ' ', dataset['review'][i])
    review = review.lower()
    review = review.split()

    review = [lemmatizer.lemmatize(word) for word in review if word not in stop_words]
    review = ' '.join(review)
    corpus.append(review)

In [4]:
len(corpus)

50000

In [5]:
from tensorflow.keras.preprocessing.text import one_hot
vocab_size=7000

2025-08-25 18:44:38.499566: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-25 18:44:38.503179: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-25 18:44:38.513111: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756129478.529361   50701 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756129478.534017   50701 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756129478.545874   50701 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [6]:
ohe=[one_hot(words,vocab_size) for words in corpus]
ohe

[[5076,
  1336,
  6075,
  2737,
  2568,
  953,
  5303,
  2810,
  2220,
  3647,
  1962,
  4646,
  4646,
  969,
  284,
  1953,
  953,
  3907,
  3148,
  2851,
  1853,
  3151,
  2220,
  510,
  6970,
  6743,
  2218,
  148,
  1344,
  4107,
  2218,
  6577,
  306,
  2553,
  1143,
  2681,
  1853,
  5177,
  2733,
  3366,
  510,
  4646,
  4646,
  3683,
  953,
  1376,
  1312,
  6600,
  3042,
  2973,
  469,
  2029,
  1647,
  710,
  6666,
  4980,
  6093,
  1561,
  5813,
  117,
  32,
  1879,
  4328,
  5973,
  6097,
  4167,
  6958,
  3126,
  4980,
  234,
  3858,
  5533,
  6308,
  19,
  2975,
  6698,
  2338,
  6602,
  6837,
  6211,
  1514,
  2683,
  6222,
  5804,
  1858,
  6878,
  523,
  190,
  4646,
  4646,
  3923,
  2846,
  3886,
  27,
  2218,
  3,
  1931,
  6970,
  2218,
  1642,
  5144,
  3275,
  5597,
  2112,
  2649,
  3429,
  5144,
  1914,
  5144,
  2985,
  953,
  4180,
  2417,
  969,
  5303,
  1785,
  3787,
  1953,
  4784,
  353,
  2846,
  3896,
  5547,
  2918,
  2468,
  953,
  2353,
  4273,
  41

## Padding now:

In [32]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dense,LSTM,Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [8]:
max_sent=50
padding=pad_sequences(ohe,maxlen=max_sent,padding='post')
padding

array([[ 353, 2846, 3896, ..., 3312, 4207, 6689],
       [2777, 2737,  753, ..., 6094, 6188, 4739],
       [1222, 1388, 6986, ..., 6970,  643, 2359],
       ...,
       [5192, 2007,  290, ..., 1474, 5022, 2189],
       [2022, 1921, 2515, ...,  967, 1884,  469],
       [2164, 6864,  523, ..., 2695, 5119, 5561]], dtype=int32)

In [33]:
model=Sequential()
model.add(Embedding(vocab_size,100,input_length=max_sent))
model.add(Dropout(0.2))
model.add(LSTM(200))
model.add(Dense(1,activation='sigmoid'))
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

/home/hammadali08/.local/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [34]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
Y=pd.get_dummies(dataset['sentiment'],drop_first=True)
Y=Y.astype('int64')
Y

,positive
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


In [36]:
import numpy as np
X=np.array(padding)
Y=np.array(Y)

In [37]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=0)

# Model Training

In [38]:
model.fit(x_train,y_train,epochs=10,batch_size=50,validation_data=(x_test,y_test),callbacks=EarlyStopping())

Epoch 1/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 60s 72ms/step - accuracy: 0.8068 - loss: 0.4194 - val_accuracy: 0.8392 - val_loss: 0.3694
Epoch 2/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 62s 78ms/step - accuracy: 0.8638 - loss: 0.3193 - val_accuracy: 0.8373 - val_loss: 0.3664
Epoch 3/10
800/800 ━━━━━━━━━━━━━━━━━━━━ 69s 86ms/step - accuracy: 0.8919 - loss: 0.2645 - val_accuracy: 0.8304 - val_loss: 0.3888


In [39]:
y_pred=model.predict(x_test)
y_pred=np.where(y_pred>0.7,1,0) # AUC ROC Curve

313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step


In [44]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred)

array([[4507,  528],
       [1340, 3625]])

In [45]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.77      0.90      0.83      5035
           1       0.87      0.73      0.80      4965

    accuracy                           0.81     10000
   macro avg       0.82      0.81      0.81     10000
weighted avg       0.82      0.81      0.81     10000



In [46]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,y_pred)

0.8126224184985064

In [47]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.8132